In [0]:
/*
WITH percentiles AS (
    SELECT 
        PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY TOT_SALES) AS p1,
        PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY TOT_SALES) AS p99
    FROM transactions
)
SELECT *
FROM transactions, percentiles
WHERE TOT_SALES < p1 OR TOT_SALES > p99;

-- Checking Outliers using standard deviation.
WITH stats AS (
    SELECT 
        AVG(TOT_SALES) AS mean,
        STDDEV(TOT_SALES) AS stddev
    FROM transactions
)
SELECT *
FROM transactions, stats
WHERE ABS(TOT_SALES - mean) > 3 * stddev;

-- another code for standard deviation 
Select * From transactions
 WHERE TOT_SALES >   ( 
    SELECT 
        AVG(TOT_SALES) + 3* STDDEV(TOT_SALES)
    FROM transactions)
OR TOT_SALES < (
    SELECT 
        AVG(TOT_SALES) - 3* STDDEV(TOT_SALES)
    FROM transactions
)
*/


-- Desctripitve Statistics
Select * from transactions;

with Amount as (
      Select 
        transaction_qty * unit_price
    From transactions
)
Select  
    Count(*) as total_transactions,
    Round(Avg(Amount),2) as Mean_Amount, -- Where Amount = transaction_qty * unit_price
    Round(Stddev(Amount),2) as Stddev_Amount,
    Round(variance(amount),2) as variance,
    Round(Min(Amount),2) as Min_Amount,
    Round(Max(Amount),2) as Max_Amount,
    Round(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY Amount),2) as Median_Amount,
    Max(Amount) - Min(Amount) as Range_Amount
From transactions;

-- Percentiles and Quartiles
Select 
    ROUND(PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY Amount),2) as Q1,
    ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY Amount),2) as Q3,
    Round(PERCENTILE_CONT(0.50) WITHIN GROUP (ORDER BY Amount),2) as Q2_Median,
    Round(PERCENTILE_CONT(0.10) WITHIN GROUP (ORDER BY Amount),2) as P10,
    Round(PERCENTILE_CONT(0.90) WITHIN GROUP (ORDER BY Amount), as P90
From transactions

-- Outlier Detection - IQR Method
With Quartiles as (
  Select
        ROUND(PERCENTILE_CONT(0.25) WITHIN GROUP (ORDER BY Amount),2) as Q1,
        ROUND(PERCENTILE_CONT(0.75) WITHIN GROUP (ORDER BY Amount),2) as Q3
        From transactions
), 
IQR_Calculation as (
  Select 
      q1, q3, (q3-q1) as IQR
  From Quartiles
)
Select COUNT(Case when Amount < q1 - 1.5*iqr then 1 end) as Lower_Outliers,
       COUNT(Case when Amount > q3 + 1.5*iqr then 1 end) as Upper_Outliers, -- we can then add Lower_Outliers + Upper_Outliers as Total_Outliers
       ROUND (AVG(CASE WHEN Amount >= (q1 - 1.5*iqr) and Amount <= (q3 - 1.5*iqr) THEN Amount END) as Mean_without_Outliers
From transactions, IQR_Calculation;

-- Outlier Detection - Z-Score Method
WITH stats AS (
    SELECT 
        AVG(Amount) AS mean,
        STDDEV(Amount) AS stddev
    FROM transactions
)
SELECT 
    transaction_id, amount, 
    Round((Amount - mean)/stddev,2) as z_score,
CASE  
    WHEN ABS((Amount - mean)/stddev) > 3 then 'Extreme / Outlier'
    WHEN ABS((Amount - mean)/stddev) > 2 then 'Moderate'
    ELSE 'Normal'
END AS Anomaly_Flag
FROM transactions, stats;

-- Check Times
Select 
  transaction_date,
  max(transaction_time) as closing_time,
  min(transaction_time) as opening_time
 from transactions
 group by transaction_date
 Limit 7

-- checking the transaction times to ascertain what time this shop opens and closes
 Select transaction_date,transaction_time from transactions where transaction_date = '2023-01-01'